## 1. Load Processed Dataset

Load the preprocessed HateXplain dataset.


In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/hatexplain_preprocessed.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (19229, 12)


,post_id,text,annotator_labels,label_counts,final_label,is_full_agreement,is_disagreement,is_undecided,targets,rationales,source,label_id
0,1179055004553900032_twitter,i dont think im getting my baby them white 9 h...,"['normal', 'normal', 'normal']",{'normal': 3},normal,1,0,0,['None'],[],twitter,2
1,1179063826874032128_twitter,we cannot continue calling ourselves feminists...,"['normal', 'normal', 'normal']",{'normal': 3},normal,1,0,0,['None'],[],twitter,2
2,1178793830532956161_twitter,nawt yall niggers ignoring me,"['normal', 'normal', 'hatespeech']","{'normal': 2, 'hatespeech': 1}",normal,0,1,0,"['None', 'African']",[],twitter,2
3,1179088797964763136_twitter,<user> i am bit confused coz chinese ppl can n...,"['hatespeech', 'offensive', 'hatespeech']","{'hatespeech': 2, 'offensive': 1}",hatespeech,0,1,0,['Asian'],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",twitter,0
4,1179085312976445440_twitter,this bitch in whataburger eating a burger with...,"['hatespeech', 'hatespeech', 'offensive']","{'hatespeech': 2, 'offensive': 1}",hatespeech,0,1,0,"['Women', 'Caucasian']","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",twitter,0


## 2. Train / Validation / Test Split

Split dataset into train, validation, and test sets using stratified sampling.


In [2]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_id"],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)


Train shape: (15383, 12)
Validation shape: (1923, 12)
Test shape: (1923, 12)


## 3. Verify Class Distribution

Ensure class distribution is preserved across splits.


In [3]:
print("Full dataset:")
print(df["final_label"].value_counts(normalize=True) * 100)

print("\nTrain:")
print(train_df["final_label"].value_counts(normalize=True) * 100)

print("\nValidation:")
print(val_df["final_label"].value_counts(normalize=True) * 100)

print("\nTest:")
print(test_df["final_label"].value_counts(normalize=True) * 100)


Full dataset:
final_label
normal        40.636539
hatespeech    30.864840
offensive     28.498622
Name: proportion, dtype: float64

Train:
final_label
normal        40.635767
hatespeech    30.865241
offensive     28.498992
Name: proportion, dtype: float64

Validation:
final_label
normal        40.665627
hatespeech    30.837233
offensive     28.497140
Name: proportion, dtype: float64

Test:
final_label
normal        40.613625
hatespeech    30.889236
offensive     28.497140
Name: proportion, dtype: float64


## 4. Save Splits

Save dataset splits for reproducibility.


In [4]:
import os

os.makedirs("../data/splits", exist_ok=True)

train_df.to_csv("../data/splits/train.csv", index=False)
val_df.to_csv("../data/splits/val.csv", index=False)
test_df.to_csv("../data/splits/test.csv", index=False)

print("Splits saved successfully.")


Splits saved successfully.


## 5. Load BERT Tokenizer

Load the pretrained BERT tokenizer.


In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


## 6. Test Tokenization

Test tokenizer on one sample.


In [6]:
sample_text = train_df["text"].iloc[0]

encoded_sample = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Original text:")
print(sample_text)

print("\nTokenized keys:")
print(encoded_sample.keys())

print("\nInput IDs shape:", encoded_sample["input_ids"].shape)
print("Attention mask shape:", encoded_sample["attention_mask"].shape)


Original text:
next time a nigga hits me up has a gf i will screenshot tf out of it send it to her you all niggas triflin

Tokenized keys:
KeysView({'input_ids': tensor([[  101,  2279,  2051,  1037,  9152, 23033,  4978,  2033,  2039,  2038,
          1037,  1043,  2546,  1045,  2097, 12117, 12326,  1056,  2546,  2041,
          1997,  2009,  4604,  2009,  2000,  2014,  2017,  2035,  9152, 23033,
          2015, 13012, 10258,  2378,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     

## 7. Tokenize Full Dataset

Convert all text into tokenized format for BERT.


In [7]:
train_encodings = tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization complete.")


Tokenization complete.


## 8. Prepare Labels

Extract label IDs for training.


In [8]:
train_labels = train_df["label_id"].tolist()
val_labels = val_df["label_id"].tolist()
test_labels = test_df["label_id"].tolist()

print(train_labels[:5])


[2, 1, 0, 2, 0]


## 9. Create PyTorch Dataset Class

Wrap encodings and labels into a PyTorch dataset.


In [9]:
import torch

class HateXplainDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


## 10. Create Dataset Objects

Create dataset objects for training, validation, and testing.


In [10]:
train_dataset = HateXplainDataset(train_encodings, train_labels)
val_dataset = HateXplainDataset(val_encodings, val_labels)
test_dataset = HateXplainDataset(test_encodings, test_labels)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))


Train size: 15383
Validation size: 1923
Test size: 1923


## 11. Import Training Libraries

Import transformers and metrics libraries for BERT training.


In [11]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np


## 12. Initialize BERT Model

Load a pretrained BERT model for sequence classification with 3 labels.


In [12]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

print("Model loaded successfully")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully


## 13. Define Metrics Function

Create a function to compute accuracy, precision, recall, and F1 score during evaluation.


In [13]:
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 score."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


## 14. Define Training Arguments

Configure hyperparameters and training settings.


In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="tensorboard"
)

print("Training arguments configured")

Training arguments configured


## 15. Initialize Trainer

Create a Trainer object to manage training, evaluation, and model checkpointing.


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer initialized")


Trainer initialized


In [16]:
# import os
# print(os.listdir("./results"))

## 16. Train Model

Start the training process. This will take some time depending on your hardware.


In [17]:
print("Starting training...")
trainer.train()


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.713344,0.760552,0.683827,0.675519,0.683827,0.673078
2,0.622286,0.716233,0.691628,0.694407,0.691628,0.690745
3,0.412607,0.873614,0.691628,0.690685,0.691628,0.690391
4,0.293148,1.428986,0.682267,0.686180,0.682267,0.683106
5,0.206669,1.878910,0.673947,0.676624,0.673947,0.673088
6,0.147850,2.132954,0.687467,0.683141,0.687467,0.684997
7,0.063550,2.483099,0.660426,0.669116,0.660426,0.661551
8,0.021538,2.720985,0.668747,0.679554,0.668747,0.672172
9,0.005898,2.811901,0.670307,0.678425,0.670307,0.672908
10,0.013308,2.910112,0.670827,0.677062,0.670827,0.672640


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=19230, training_loss=0.26649305638065346, metrics={'train_runtime': 4259.6465, 'train_samples_per_second': 36.113, 'train_steps_per_second': 4.514, 'total_flos': 1.011868426227456e+16, 'train_loss': 0.26649305638065346, 'epoch': 10.0})

In [18]:
# from transformers import BertForSequenceClassification, Trainer, TrainingArguments

# model = BertForSequenceClassification.from_pretrained("./results/checkpoint-3846")

# training_args = TrainingArguments(
#     output_dir="./results",
#     per_device_eval_batch_size=8,
#     report_to="none",
#     disable_tqdm=True,
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics,
# )

## 17. Evaluate Model

Evaluate the trained model on validation and test sets.


In [19]:
# print("Validation results:")
# val_results = trainer.evaluate(eval_dataset=val_dataset)
# print(val_results)

# print("\nTest results:")
# test_results = trainer.evaluate(eval_dataset=test_dataset)
# print(test_results)


## 18. Save Trained Model

Save the trained model and tokenizer for future use.


In [20]:
model.save_pretrained("./saved_model2")
tokenizer.save_pretrained("./saved_model2")

print("Model and tokenizer saved to ./saved_model2")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./saved_model2
